In [1]:
import numpy as np
import astropy.units as u
from astropy import constants as const

def solid_angle(r):
    """
    Calculates the solid angle of the source.

    Parameters:
    - r: Radius of the source (in degrees)

    Returns:
    - Calculated solid angle of the source.
    """
    return (2 * np.pi * (1 - np.cos(r * u.deg))).to(u.sr)

In [ ]:
def mol_upper_state_cd(A_ul, int_flux, dx, dy, th_maj, th_min, r):
    """
    Calculates the molecular upper state column density.

    N_u = (4π / (A_ul * Ω * h * c)) * ∫I_ν dV * (dx*dy * 4ln2) / (π * θ_maj * θ_min)

    Parameters:
    - A_ul:      Einstein A coefficient [s^-1]
    - int_flux:  Integrated flux ∫I_ν dV [Jy * km/s]
    - dx, dy:    Angular pixel/cell size [arcsec]
    - th_maj:    Beam major axis FWHM [arcsec]
    - th_min:    Beam minor axis FWHM [arcsec]
    - r:         Radius of the source for solid angle calculation [deg]

    Returns:
    - N_u:       Upper state column density [cm^-2]
    """
    omega = solid_angle(r).value 
    arcsec_to_rad = (1 * u.arcsec).to(u.rad).value
    omega_pixel = dx * dy * arcsec_to_rad**2
    omega_beam = np.pi * th_maj * th_min * arcsec_to_rad**2 / (4 * np.log(2))
    beam_corr = omega_pixel / omega_beam
    int_flux_cgs = int_flux * 1e-23 * 1e5
    h = const.h.cgs.value   # erg*s
    c = const.c.cgs.value   # cm/s
    N_u = (4 * np.pi * int_flux_cgs * beam_corr) / (A_ul * omega * h * c)

    return N_u  # cm^-2

In [5]:
def total_mol_cd(N_u, g_u, Q_rot, E_u, T_rot):
    """
    Calculates the total molecular column density.

    N_T = N_u * Q(T_rot) * e^(E_u / T_rot) / g_u

    Parameters:
    - N_u:   Upper state column density [cm^-2], from mol_upper_state_cd()
    - g_u:   Upper state degeneracy [dimensionless]
    - Q_rot: Partition function at T_rot [dimensionless]
    - E_u:   Upper state energy [K]  (E_u/k_B, already in temperature units)
    - T_rot: Rotational temperature [K]

    Returns:
    - N_T:   Total molecular column density [cm^-2]
    """
    N_T = (N_u * Q_rot * np.exp(E_u / T_rot)) / g_u
    return N_T  # cm^-2